# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('teleport')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[2 1]'; Monster is at '[1 1]'
.....
.M...
.H...
.....
.....
Hunter chose action 'west'
Monster chose action 'west'


Step 2: Hunter is at '[2 0]'; Monster is at '[1 0]'
.....
M....
H....
.....
.....
Hunter chose action 'east'
Monster chose action 'north'


Step 3: Hunter is at '[2 1]'; Monster is at '[0 0]'
M....
.....
.H...
.....
.....
Hunter chose action 'listen'
Monster chose action 'stay'


Step 4: Hunter is at '[2 1]'; Monster is at '[0 0]'
M....
.....
.H...
.....
.....
Hunter chose action 'west'
Monster chose action 'stay'


Step 5: Hunter is at '[2 0]'; Monster is at '[0 0]'
M....
.....
H....
.....
.....
Hunter chose action 'east'
Monster chose action 'stay'


Step 6: Hunter is at '[2 1]'; Monster is at '[0 0]'
M....
.....
.H...
.....
.....
Hunter chose action 'listen'
Monster chose action 'stay'


Step 7: Hunter is at '[2 1]'; Monster is at '[0 0]'
M....
.....
.H...
.....
.....
Hunter chose action 'teleport'
Monster chose action 'stay'


Step 8: Hunter is 

8

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [7]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[nan, nan, 42, 44, 45, nan, 4, nan, nan, nan, nan, nan, 35, 8, 3, 44, nan, 26, nan, 16, nan, nan, nan, nan, nan, nan, 44, 28, 75, 89, 0, 75, nan, 17, 14, 85, 11, 8, 0, 27, 14, nan, nan, 17, nan, 13, nan, 3, nan, nan, nan, 71, 30, 78, nan, 13, 28, nan, nan, 26, 55, nan, 7, nan, 91, 62, nan, nan, 80, nan, nan, 24, nan, 42, nan, 93, 88, nan, 30, nan, nan, 42, nan, nan, nan, nan, 94, 15, nan, nan, 98, nan, 51, 56, 73, 44, nan, 0, 2, 63]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [8]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 46 times.
Average steps to catch the monster over 54 successful runs: 39.69


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

**Thiết kế của em:** đây là một *model-based reflex agent* — nó cần nhớ **vị trí quái vật nghe được lần gần nhất** (`target`) trong một trạng thái nội bộ, vì môi trường chỉ cho biết vị trí quái vật đúng ngay sau khi hunter vừa `listen` (các bước khác `monster_location` sẽ là `None`). Chiến lược: nếu chưa có thông tin gì → `listen`; nếu đã đứng đúng tại vị trí nghe được gần nhất mà vẫn chưa bắt được (nghĩa là quái vật đã di chuyển đi nơi khác) → `listen` lại để cập nhật; còn lại thì di chuyển theo hướng làm giảm khoảng cách Manhattan tới vị trí quái vật đã biết (ưu tiên trục có khoảng cách lệch lớn hơn trước). Vì dùng closure để lưu `state` riêng, mỗi lần chạy mô phỏng mới cần gọi `make_hunter_agent()` để tạo một agent với bộ nhớ hoàn toàn mới.

In [9]:
def make_hunter_agent():
    """
    Factory that returns a FRESH model-based reflex hunter agent (its own
    private memory of the monster's last known location). Call this once per
    simulation run: arena_environment(make_hunter_agent(), monster_fn, ...).
    """
    state = {"target": None}

    def agent(hunter_location, monster_location):
        hp = tuple(int(v) for v in hunter_location)

        # this is feedback from a 'listen' action taken in the previous step
        if monster_location is not None:
            state["target"] = tuple(int(v) for v in monster_location)

        target = state["target"]

        # no information yet -> the only way to get it is to listen
        if target is None:
            return "listen"

        # we reached where we last heard the monster but did not catch it
        # -> it must have moved away meanwhile, so listen again to relocate it
        if hp == target:
            return "listen"

        # otherwise: greedily move towards the last known location
        # (reduce the larger of the two coordinate differences first)
        row_diff = target[0] - hp[0]
        col_diff = target[1] - hp[1]
        if abs(row_diff) >= abs(col_diff):
            return "south" if row_diff > 0 else "north"
        else:
            return "east" if col_diff > 0 else "west"

    return agent


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

Em minh hoạ một lượt chạy có hiển thị (`visualize=True`), rồi chạy thực nghiệm 100 lần trên hai kích thước đấu trường $10 \times 10$ và $30 \times 30$, báo cáo tỷ lệ bắt thành công và số bước trung bình, theo đúng định dạng đã dùng ở ví dụ hunter ngẫu nhiên phía trên.

In [10]:
# Demo: one visualized run of our hunter agent
np.random.seed(11)
arena_environment(make_hunter_agent(), monster_agent_function_simple,
                   n=5, max_steps=20, visualize=True, animation=False)


Step 1: Hunter is at '[0 3]'; Monster is at '[1 3]'
...H.
...M.
.....
.....
.....
Hunter chose action 'listen'
Monster chose action 'south'


Step 2: Hunter is at '[0 3]'; Monster is at '[2 3]'
...H.
.....
...M.
.....
.....
Hunter chose action 'south'
Monster chose action 'stay'


Step 3: Hunter is at '[1 3]'; Monster is at '[2 3]'
.....
...H.
...M.
.....
.....
Hunter chose action 'south'
Monster chose action 'south'


Step 4: Hunter is at '[2 3]'; Monster is at '[3 3]'
.....
.....
...H.
...M.
.....
Hunter chose action 'listen'
Monster chose action 'south'


Step 5: Hunter is at '[2 3]'; Monster is at '[4 3]'
.....
.....
...H.
.....
...M.
Hunter chose action 'south'
Monster chose action 'north'


Step 6: Hunter is at '[3 3]'; Monster is at '[3 3]'
.....
.....
.....
...*.
.....
Hunter caught the monster in 5 steps!


5

In [11]:
def evaluate_hunter(hunter_factory, monster_function, n, max_steps=100, runs=100):
    """Run `runs` simulations and return the array of steps-to-catch (np.nan = failed)."""
    steps = [arena_environment(hunter_factory(), monster_function,
                                n=n, max_steps=max_steps, visualize=False)
             for _ in range(runs)]
    return np.array(steps, dtype=float)


np.random.seed(12)
for n in (10, 30):
    steps = evaluate_hunter(make_hunter_agent, monster_agent_function_simple, n=n,
                             max_steps=100, runs=100)
    failed = int(np.sum(np.isnan(steps)))
    print(f"Arena {n}x{n} ({100 - failed}/100 successful runs):")
    print(f"  average steps to catch the monster: {np.nanmean(steps):.2f}")
    print(f"  hunter failed {failed} times out of 100 (success rate {100 - failed}%)\n")


Arena 10x10 (100/100 successful runs):
  average steps to catch the monster: 10.07
  hunter failed 0 times out of 100 (success rate 100%)

Arena 30x30 (100/100 successful runs):
  average steps to catch the monster: 26.08
  hunter failed 0 times out of 100 (success rate 100%)



## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

**Chiến lược cuối cùng của hunter và vì sao nó hiệu quả:**
Hunter của em là một *model-based reflex agent*: nó ghi nhớ vị trí quái vật nghe được lần gần nhất
(`target`) rồi liên tục di chuyển theo hướng giảm khoảng cách Manhattan đến vị trí đó (ưu tiên trục
lệch nhiều hơn). Nó chỉ `listen` lại khi chưa có thông tin, hoặc khi đã đến đúng vị trí cũ mà chưa
bắt được (nghĩa là quái vật đã dịch chuyển trong lúc hunter tới gần). Chiến lược này hiệu quả vì:
quái vật phần lớn thời gian đứng yên ($p=0.5$ chọn `stay`), nên một lần nghe là đủ thông tin đáng
tin cậy cho rất nhiều bước di chuyển tiếp theo — hunter không lãng phí lượt vào việc nghe liên tục,
mà dồn gần như toàn bộ số bước cho việc tiến thẳng tới mục tiêu.

**Có dùng teleport không?**
Em **không dùng** `teleport` trong chiến lược chính. Vì hunter luôn biết chính xác hướng cần đi
(nhờ `listen`), di chuyển từng bước một theo hướng đó đã là gần tối ưu (chỉ tốn khoảng bằng đúng
khoảng cách Manhattan tới quái vật, cộng thêm một vài lần `listen` để hiệu chỉnh) — `teleport` đưa
hunter đến một ô **ngẫu nhiên** trong đấu trường, có thể đẩy hunter ra xa quái vật hơn tình trạng
hiện tại, nên khi đã có thông tin định vị tốt, teleport chỉ làm giảm hiệu năng trung bình. Teleport
sẽ hữu ích hơn trong tình huống hunter **hoàn toàn không có thông tin** (ví dụ nếu không được phép
`listen`, hoặc cần thoát khỏi một góc kẹt) — khi đó teleport đóng vai trò như một phép "reset" vị trí
ngẫu nhiên, tương tự việc bắt đầu lại việc dò tìm.

**Kích thước đấu trường ảnh hưởng thế nào đến hiệu năng:**
Số liệu thực nghiệm cho thấy tỷ lệ bắt thành công của hunter gần như không đổi (rất cao) khi tăng
kích thước đấu trường từ $10\times10$ lên $30\times30$, nhưng **số bước trung bình để bắt được quái
vật tăng lên** theo kích thước — hợp lý vì khoảng cách Manhattan trung bình giữa hai vị trí ngẫu
nhiên trên lưới tỉ lệ thuận với $n$ (kích thước cạnh), nên hunter cần nhiều bước di chuyển hơn để
tới nơi. Điều này khác hẳn với hunter ngẫu nhiên hoàn toàn ở ví dụ ban đầu — hunter đó thất bại rất
nhiều ngay cả ở $10\times10$ vì nó không hề dùng thông tin định vị, còn hunter có bộ nhớ của em vẫn
giữ được hiệu năng tốt khi bài toán mở rộng quy mô, đúng như bài học rút ra ở phần Robot hút bụi.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

Quái vật hiện tại chọn hành động hoàn toàn không phụ thuộc vào vị trí hunter (`monster_agent_function_simple`
bỏ qua cả hai tham số, chỉ chọn ngẫu nhiên có trọng số cố định). Để "giỏi hơn" trong việc né hunter,
quái vật cần trở thành một **agent phản xạ dựa trên percept vị trí hunter** — tức là nó phải dùng
chính tham số `hunter_pos` được truyền vào:

* **Phản xạ né tránh đơn giản:** nếu khoảng cách tới hunter nhỏ hơn một ngưỡng, chủ động di chuyển
  theo hướng **làm tăng** khoảng cách Manhattan tới hunter (ngược hoàn toàn với chiến lược của hunter
  ở trên), thay vì chọn ngẫu nhiên/đứng yên như hiện tại.
* **Ưu tiên đứng yên khi an toàn:** khi hunter còn ở xa, vẫn có thể giữ xác suất `stay` cao để khó bị
  hunter "nghe" thấy dấu hiệu di chuyển thường xuyên (dù trong bài này percept của hunter không phụ
  thuộc chuyển động của quái vật, đây vẫn là một hướng thiết kế hợp lý về nguyên tắc agent).
* **Yếu tố ngẫu nhiên có kiểm soát:** vẫn giữ một phần ngẫu nhiên trong lựa chọn hướng né để tránh bị
  hunter dự đoán chính xác 100% quy luật di chuyển (nếu quái vật né theo một công thức xác định tuyệt
  đối, hunter học được công thức đó có thể dự đoán trước và chặn đường).

Đây chính là một agent **model-free nhưng phản xạ có điều kiện dựa trên percept hunter_pos** — nâng
cấp so với agent ngẫu nhiên/tĩnh hiện tại, và sẽ làm giảm tỷ lệ bắt thành công cũng như tăng số bước
trung bình của hunter trong các thực nghiệm ở trên.

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.